# Data Processing — Employee Performance at INX Future Inc

Purpose: load the raw employee data, validate its structure, clean it, and save a
ready-to-use version for analysis and modelling.

## 1.Imports and Setup

In [2]:
import sys
!{sys.executable} -m pip install xlrd

In [4]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

## 2. Load the Raw Data

In [5]:
raw_path = "../../data/raw/INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1.8.xls"

df = pd.read_excel(raw_path)
print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (1200, 28)


,EmpNumber,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,OverTime,EmpLastSalaryHikePercent,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,PerformanceRating
0,E1001000,32,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,10,3,4,55,3,2,4,1,No,12,4,10,2,2,10,7,0,8,No,3
1,E1001006,47,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,14,4,4,42,3,2,1,2,No,12,4,20,2,3,7,7,1,7,No,3
2,E1001007,40,Male,Life Sciences,Married,Sales,Sales Executive,Travel_Frequently,5,4,4,48,2,3,1,5,Yes,21,3,20,2,3,18,13,1,12,No,4
3,E1001009,41,Male,Human Resources,Divorced,Human Resources,Manager,Travel_Rarely,10,4,2,73,2,5,4,3,No,15,2,23,2,2,21,6,12,6,No,3
4,E1001010,60,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,16,4,1,84,3,2,1,8,No,14,4,10,1,3,2,2,2,2,No,3


## 3. Structural Check

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   EmpNumber                     1200 non-null   object
 1   Age                           1200 non-null   int64 
 2   Gender                        1200 non-null   object
 3   EducationBackground           1200 non-null   object
 4   MaritalStatus                 1200 non-null   object
 5   EmpDepartment                 1200 non-null   object
 6   EmpJobRole                    1200 non-null   object
 7   BusinessTravelFrequency       1200 non-null   object
 8   DistanceFromHome              1200 non-null   int64 
 9   EmpEducationLevel             1200 non-null   int64 
 10  EmpEnvironmentSatisfaction    1200 non-null   int64 
 11  EmpHourlyRate                 1200 non-null   int64 
 12  EmpJobInvolvement             1200 non-null   int64 
 13  EmpJobLevel       

In [7]:
df.isna().sum()

EmpNumber                       0
Age                             0
Gender                          0
EducationBackground             0
MaritalStatus                   0
EmpDepartment                   0
EmpJobRole                      0
BusinessTravelFrequency         0
DistanceFromHome                0
EmpEducationLevel               0
EmpEnvironmentSatisfaction      0
EmpHourlyRate                   0
EmpJobInvolvement               0
EmpJobLevel                     0
EmpJobSatisfaction              0
NumCompaniesWorked              0
OverTime                        0
EmpLastSalaryHikePercent        0
EmpRelationshipSatisfaction     0
TotalWorkExperienceInYears      0
TrainingTimesLastYear           0
EmpWorkLifeBalance              0
ExperienceYearsAtThisCompany    0
ExperienceYearsInCurrentRole    0
YearsSinceLastPromotion         0
YearsWithCurrManager            0
Attrition                       0
PerformanceRating               0
dtype: int64

In [8]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate EmpNumber:", df["EmpNumber"].duplicated().sum())

Duplicate rows: 0
Duplicate EmpNumber: 0


## 4. Target Column Check

In [9]:
df["PerformanceRating"].value_counts().sort_index()

PerformanceRating
2    194
3    874
4    132
Name: count, dtype: int64

### Observations

- The dataset has **1,200 rows and 28 columns** — one row per employee.
- **No missing values** anywhere, and **no duplicate rows or duplicate `EmpNumber` values** — each employee appears exactly once, and no imputation is needed.
- `PerformanceRating` contains three values: **2 (Good) = 194 employees, 3 (Excellent) = 874, 4 (Outstanding) = 132**. No rating of 1 is present.
- The classes are **imbalanced**: rating 3 alone makes up 72.8% of employees. A model that always predicted "3" would already be right ~73% of the time — so **accuracy alone will not be a meaningful success measure** later; macro F1 and per-class recall will matter more.
- 19 numeric columns, 9 text (object) columns. Text columns still need a whitespace/label consistency check before use.

## 5. Text Column Consistency Check

Even with no missing values, category labels can have inconsistent spelling, spacing,
or casing. This step inspects each text column.

In [11]:
text_cols = df.select_dtypes(include="object").columns.tolist()
text_cols.remove("EmpNumber")   # identifier, not a category

for col in text_cols:
    print(f"\n{col} ({df[col].nunique()} unique values):")
    print(sorted(df[col].unique()))


Gender (2 unique values):
['Female', 'Male']

EducationBackground (6 unique values):
['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']

MaritalStatus (3 unique values):
['Divorced', 'Married', 'Single']

EmpDepartment (6 unique values):
['Data Science', 'Development', 'Finance', 'Human Resources', 'Research & Development', 'Sales']

EmpJobRole (19 unique values):
['Business Analyst', 'Data Scientist', 'Delivery Manager', 'Developer', 'Finance Manager', 'Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manager R&D', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative', 'Senior Developer', 'Senior Manager R&D', 'Technical Architect', 'Technical Lead']

BusinessTravelFrequency (3 unique values):
['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']

OverTime (2 unique values):
['No', 'Yes']

Attrition (2 unique values):
['No', 'Yes']


### Observations

- All text columns have clean, consistent category labels — no whitespace issues, casing
  mismatches, or duplicate-meaning categories.
- `EmpDepartment` has 6 categories; `EmpJobRole` has 19, nested under departments (e.g.
  Sales Executive / Sales Representative under Sales). Some job roles will have small
  headcounts, so role-level averages should be read cautiously.
- No cleaning action needed on text columns.

## 6. Safety-Net Cleaning

No issues were found above, but this step is kept as a safeguard in case the dataset
changes later (e.g. a refreshed export with new rows).

In [12]:
before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="EmpNumber")
print("Rows removed:", before - len(df))

for col in df.columns:
    if df[col].isna().sum() > 0:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode().iloc[0])
print("Missing values remaining:", df.isna().sum().sum())

Rows removed: 0
Missing values remaining: 0


## 7. Feature Engineering

The tenure-related columns overlap heavily (e.g. years in current role can't exceed
years at the company), so their raw values mostly repeat the same "length of service"
signal. Ratio features below expose different information: how someone's tenure is
*shaped*, not just how long it is.

| New feature | Formula | What it captures |
|---|---|---|
| `PromotionGapRatio` | years since last promotion / (years at company + 1) | Share of tenure spent without a promotion |
| `RoleTenureRatio` | years in current role / (years at company + 1) | Stagnation in the same role |
| `ManagerTenureRatio` | years with current manager / (years at company + 1) | Management stability |
| `PriorExperienceYears` | total experience − years at company | Experience brought in from outside |
| `AverageTenurePerEmployer` | total experience / (companies worked + 1) | Job-hopping tendency |

In [13]:
tenure = df["ExperienceYearsAtThisCompany"] + 1

df["PromotionGapRatio"] = (df["YearsSinceLastPromotion"] / tenure).round(4)
df["RoleTenureRatio"] = (df["ExperienceYearsInCurrentRole"] / tenure).round(4)
df["ManagerTenureRatio"] = (df["YearsWithCurrManager"] / tenure).round(4)
df["PriorExperienceYears"] = (df["TotalWorkExperienceInYears"] - df["ExperienceYearsAtThisCompany"]).clip(lower=0)
df["AverageTenurePerEmployer"] = (df["TotalWorkExperienceInYears"] / (df["NumCompaniesWorked"] + 1)).round(3)

df[["PromotionGapRatio", "RoleTenureRatio", "ManagerTenureRatio",
    "PriorExperienceYears", "AverageTenurePerEmployer"]].describe()

,PromotionGapRatio,RoleTenureRatio,ManagerTenureRatio,PriorExperienceYears,AverageTenurePerEmployer
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,0.236302,0.485773,0.461804,4.252500,4.238355
std,0.267830,0.271608,0.277937,6.170533,4.150637
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.333300,0.285700,0.000000,1.656500
50%,0.142900,0.500000,0.500000,2.000000,3.000000
75%,0.428600,0.666700,0.666700,5.000000,5.000000
max,0.916700,0.882400,0.894700,33.000000,38.000000


### Observations

- All five engineered ratios are well-behaved: no negative values, no infinities.
- `PromotionGapRatio` median is 0.14, meaning half of employees have spent 14% or less
  of their tenure without a promotion — but the max reaches 0.92, so some employees have
  gone almost their entire tenure without one.
- `PriorExperienceYears` ranges 0–33, confirming a mix of fresh joiners and highly
  experienced hires.

## 8. Save the Processed Dataset

In [14]:
RATING_LABELS = {1: "Low", 2: "Good", 3: "Excellent", 4: "Outstanding"}
df["PerformanceLabel"] = df["PerformanceRating"].map(RATING_LABELS)

out_path = "../../data/processed/employee_processed.csv"
df.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Final shape:", df.shape)

Saved: ../../data/processed/employee_processed.csv
Final shape: (1200, 34)


### Summary of Data Processing

- Started with 1,200 employees, 28 columns, no missing values, no duplicates.
- Verified all category labels are clean and consistent.
- Added 5 engineered ratio features plus a readable `PerformanceLabel` column.
- Saved 1,200 rows × 34 columns to `data/processed/employee_processed.csv`.
- Dataset is ready for exploratory analysis (department performance, factor
  association) in the next notebook.